# Tutorial 1: Completing the Schema (Lecture 8, Data Definition)

In the introduction notebook we built two tables, *customer* and *vendor*, but neither one referenced anything else. This tutorial finishes the rest of the schema, *item*, *cases*, *pallet*, *customerorder*, *order_item*, *shipment*, and *shipment_info*, and this time several of them reference each other with foreign keys, the way a real schema actually looks.

We will move quickly through anything we already covered in the introduction notebook (reading a CSV, a basic CREATE TABLE, looping INSERT statements), and slow down for what is new here, foreign keys, composite keys, and the rest of ALTER TABLE.

Run this notebook from inside the `jupyter` service, right after running the introduction notebook, since this one assumes *customer* and *vendor* already exist.

In [ ]:
import pymysql
import csv

In [ ]:
DB_HOST = "mysql"          # use "localhost" if running outside the jupyter container
DB_PORT = 3306
DB_USER = "root"
DB_PASSWORD = "OSUGoPokes"
DB_NAME = "iem4723_mysql_tutorial"

conn = pymysql.connect(
    host=DB_HOST,
    port=DB_PORT,
    user=DB_USER,
    password=DB_PASSWORD,
    database=DB_NAME,
)
cursor = conn.cursor()

Before we build anything, we drop any of this tutorial's tables that might already exist, in reverse dependency order, children before parents. Otherwise re-running this notebook from the top would fail partway through, since MySQL wont let us drop a table that something else still has a foreign key pointing at.

In [ ]:
reset_order = ["item_scratch_renamed", "item_scratch", "shipment_info", "order_item",
               "shipment", "customerorder", "pallet", "cases", "item"]

for table in reset_order:
    cursor.execute(f"DROP TABLE IF EXISTS {table};")

conn.commit()
print("Cleared any existing tutorial tables.")

## The Simple Tables

Remember how we built *customer* and *vendor* in the introduction notebook? *item*, *cases*, and *pallet* work the exact same way, they dont reference anything else, so theres nothing new here yet, just a quick recap before the interesting part.

In [ ]:
create_item_table_query = """
DROP TABLE IF EXISTS item;
CREATE TABLE item (
    Item_ID VARCHAR(15) PRIMARY KEY,
    Item_Description VARCHAR(50),
    Item_Price DECIMAL(8,2),
    Item_Weight DECIMAL(8,2),
    Item_AnnualSales INT,
    Item_StockLocation VARCHAR(10)
);
"""
for statement in create_item_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Item.csv", newline="") as f:
    item_rows = list(csv.DictReader(f))

insert_item_query = """
INSERT INTO item (Item_ID, Item_Description, Item_Price, Item_Weight, Item_AnnualSales, Item_StockLocation)
VALUES (%s, %s, %s, %s, %s, %s);
"""
for row in item_rows:
    cursor.execute(insert_item_query, (
        row["Item_ID"], row["Item_Description"], row["Item_Price"],
        row["Item_Weight"], row["Item_AnnualSales"], row["Item_StockLocation"],
    ))

conn.commit()
print(f"Loaded {len(item_rows)} rows into item.")

In [ ]:
create_cases_table_query = """
DROP TABLE IF EXISTS cases;
CREATE TABLE cases (
    Case_ID INT PRIMARY KEY,
    Item_Quantity INT,
    Case_Weight INT
);
"""
for statement in create_cases_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Cases.csv", newline="") as f:
    cases_rows = list(csv.DictReader(f))

insert_cases_query = """
INSERT INTO cases (Case_ID, Item_Quantity, Case_Weight)
VALUES (%s, %s, %s);
"""
for row in cases_rows:
    cursor.execute(insert_cases_query, (
        row["Case_ID"], row["Item_Quantity"], row["Case_Weight"],
    ))

conn.commit()
print(f"Loaded {len(cases_rows)} rows into cases.")

In [ ]:
create_pallet_table_query = """
DROP TABLE IF EXISTS pallet;
CREATE TABLE pallet (
    Pallet_ID INT PRIMARY KEY,
    Pallet_Weight INT,
    Pallet_Volume INT
);
"""
for statement in create_pallet_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Pallet.csv", newline="") as f:
    pallet_rows = list(csv.DictReader(f))

insert_pallet_query = """
INSERT INTO pallet (Pallet_ID, Pallet_Weight, Pallet_Volume)
VALUES (%s, %s, %s);
"""
for row in pallet_rows:
    cursor.execute(insert_pallet_query, (
        row["Pallet_ID"], row["Pallet_Weight"], row["Pallet_Volume"],
    ))

conn.commit()
print(f"Loaded {len(pallet_rows)} rows into pallet.")

## Adding a Foreign Key

*customerorder* is the first table that actually references another one. Every order belongs to a customer, so how do we capture that in SQL? With a `FOREIGN KEY` clause, *Customer_Num* in *customerorder* points back at *customer*. `ON UPDATE CASCADE` means a change to a customer number updates every order that references it automatically, and `ON DELETE RESTRICT` means MySQL refuses to delete a customer who still has orders on file.

In [ ]:
create_customerorder_table_query = """
DROP TABLE IF EXISTS customerorder;
CREATE TABLE customerorder (
    Order_Num INT PRIMARY KEY,
    Customer_Num VARCHAR(10),
    Order_Date DATE,
    Order_ReqShipDate DATE,
    Order_ActualShipDate DATE,
    Order_Carrier VARCHAR(50),
    FOREIGN KEY (Customer_Num) REFERENCES customer (Customer_Num)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""
for statement in create_customerorder_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/CustomerOrder.csv", newline="") as f:
    customerorder_rows = list(csv.DictReader(f))

insert_customerorder_query = """
INSERT INTO customerorder (Order_Num, Customer_Num, Order_Date, Order_ReqShipDate, Order_ActualShipDate, Order_Carrier)
VALUES (%s, %s, %s, %s, %s, %s);
"""
for row in customerorder_rows:
    cursor.execute(insert_customerorder_query, (
        row["Order_Num"], row["Customer_Num"], row["Order_Date"],
        row["Order_ReqShipDate"], row["Order_ActualShipDate"], row["Order_Carrier"],
    ))

conn.commit()
print(f"Loaded {len(customerorder_rows)} rows into customerorder.")

## A Table With Two Foreign Keys and a Concatenated Primary Key

*order_item* records how many of each item appeared on each order. No single column identifies a row here, an order can have many items and an item can appear on many orders, so the primary key is the concatenation of *Order_Num* and *Item_ID* together. Notice `PRIMARY KEY (Order_Num, Item_ID)` at the end of the statement, this is the same "add PRIMARY KEY at the end" style we saw for concatenated keys back in Lecture 5, plus two separate `FOREIGN KEY` clauses, one back to *customerorder* and one back to *item*.

In [ ]:
create_order_item_table_query = """
DROP TABLE IF EXISTS order_item;
CREATE TABLE order_item (
    Order_Num INT,
    Item_ID VARCHAR(15),
    Item_Qty INT,
    PRIMARY KEY (Order_Num, Item_ID),
    FOREIGN KEY (Order_Num) REFERENCES customerorder (Order_Num)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (Item_ID) REFERENCES item (Item_ID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""
for statement in create_order_item_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Order_Item.csv", newline="") as f:
    order_item_rows = list(csv.DictReader(f))

insert_order_item_query = """
INSERT INTO order_item (Order_Num, Item_ID, Item_Qty)
VALUES (%s, %s, %s);
"""
for row in order_item_rows:
    cursor.execute(insert_order_item_query, (
        row["Order_Num"], row["Item_ID"], row["Item_Qty"],
    ))

conn.commit()
print(f"Loaded {len(order_item_rows)} rows into order_item.")

## Referencing vendor

*shipment* works the same way as *customerorder*, except it references *vendor* instead of *customer*, every shipment comes from exactly one vendor.

In [ ]:
create_shipment_table_query = """
DROP TABLE IF EXISTS shipment;
CREATE TABLE shipment (
    Shipment_Num VARCHAR(15) PRIMARY KEY,
    Vendor_Num VARCHAR(10),
    Shipment_ReceiptDate DATE,
    Shipment_NumPallets INT,
    PurchaseOrder_Num INT,
    FOREIGN KEY (Vendor_Num) REFERENCES vendor (Vendor_Num)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""
for statement in create_shipment_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Shipment.csv", newline="") as f:
    shipment_rows = list(csv.DictReader(f))

insert_shipment_query = """
INSERT INTO shipment (Shipment_Num, Vendor_Num, Shipment_ReceiptDate, Shipment_NumPallets, PurchaseOrder_Num)
VALUES (%s, %s, %s, %s, %s);
"""
for row in shipment_rows:
    cursor.execute(insert_shipment_query, (
        row["Shipment_Num"], row["Vendor_Num"], row["Shipment_ReceiptDate"],
        row["Shipment_NumPallets"], row["PurchaseOrder_Num"],
    ))

conn.commit()
print(f"Loaded {len(shipment_rows)} rows into shipment.")

## A Table With Four Foreign Keys

*shipment_info* is the last table, and the busiest one. Each row says which case, of which item, on which pallet, arrived as part of which shipment, so it references four different tables at once:
    1. *shipment*: which shipment this row belongs to
    2. *cases*: which case
    3. *item*: which item was in that case
    4. *pallet*: which pallet it arrived on

A table can have as many `FOREIGN KEY` clauses as it needs, one per relationship.

In [ ]:
create_shipment_info_table_query = """
DROP TABLE IF EXISTS shipment_info;
CREATE TABLE shipment_info (
    Shipment_Num VARCHAR(15),
    Case_ID INT,
    Item_ID VARCHAR(15),
    Pallet_ID INT,
    Item_Qty INT,
    PRIMARY KEY (Shipment_Num, Case_ID),
    FOREIGN KEY (Shipment_Num) REFERENCES shipment (Shipment_Num)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (Case_ID) REFERENCES cases (Case_ID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (Item_ID) REFERENCES item (Item_ID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT,
    FOREIGN KEY (Pallet_ID) REFERENCES pallet (Pallet_ID)
        ON UPDATE CASCADE
        ON DELETE RESTRICT
);
"""
for statement in create_shipment_info_table_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

with open("../../data/orders/Shipment_Info.csv", newline="") as f:
    shipment_info_rows = list(csv.DictReader(f))

insert_shipment_info_query = """
INSERT INTO shipment_info (Shipment_Num, Case_ID, Item_ID, Pallet_ID, Item_Qty)
VALUES (%s, %s, %s, %s, %s);
"""
for row in shipment_info_rows:
    cursor.execute(insert_shipment_info_query, (
        row["Shipment_Num"], row["Case_ID"], row["Item_ID"],
        row["Pallet_ID"], row["Item_Qty"],
    ))

conn.commit()
print(f"Loaded {len(shipment_info_rows)} rows into shipment_info.")

## Checking the Whole Schema

The schema is complete now, nine tables total. Lets confirm every table loaded correctly by counting rows in each one.

In [ ]:
all_tables = ["customer", "vendor", "item", "cases", "pallet",
              "customerorder", "order_item", "shipment", "shipment_info"]

for table in all_tables:
    cursor.execute(f"SELECT COUNT(*) FROM {table};")
    count = cursor.fetchone()[0]
    print(f"{table}: {count} rows")

## Practicing ALTER TABLE

Lecture 8 also covered `ALTER TABLE`, adding columns, changing a column's definition, changing defaults and null status, deleting columns and constraints, and renaming or dropping a table entirely. We dont want to practice any of that on the real schema we just built, since later tutorials depend on it, so we will build one small scratch table just for this, a stripped-down copy of *item* called *item_scratch*.

In [ ]:
create_item_scratch_query = """
DROP TABLE IF EXISTS item_scratch;
CREATE TABLE item_scratch (
    Item_ID VARCHAR(15) PRIMARY KEY,
    Item_Description VARCHAR(50),
    Item_Price DECIMAL(8,2)
);
"""
for statement in create_item_scratch_query.strip().split(";"):
    if statement.strip():
        cursor.execute(statement)

cursor.execute("""
    INSERT INTO item_scratch (Item_ID, Item_Description, Item_Price)
    VALUES ('X001', 'Test Widget', 9.99);
""")
conn.commit()
print("item_scratch table created.")

### Adding a Column and a Named Constraint

`ADD COLUMN` and `MODIFY` work exactly as shown in lecture. We are also naming our `CHECK` constraint this time with `CONSTRAINT constraint_name`, so that we can drop that specific constraint by name later, MySQL wont let us drop an unnamed one without looking its generated name up first.

In [ ]:
cursor.execute("ALTER TABLE item_scratch ADD COLUMN Item_Notes VARCHAR(100);")
cursor.execute("ALTER TABLE item_scratch MODIFY Item_Price DECIMAL(8,2) DEFAULT 0;")
cursor.execute("ALTER TABLE item_scratch MODIFY Item_Description VARCHAR(50) NOT NULL;")
cursor.execute("""
    ALTER TABLE item_scratch
    ADD CONSTRAINT item_scratch_price_chk CHECK (Item_Price >= 0);
""")
conn.commit()

cursor.execute("DESCRIBE item_scratch;")
for row in cursor.fetchall():
    print(row)

### Deleting Columns and Constraints

**A heads-up here.** The lecture slides show the generic SQL standard keyword `DELETE` for removing a column, a check, a primary key, or a foreign key, for example `ALTER TABLE order_line DELETE line_cost;`. MySQL does not actually accept `DELETE` in that spot, it wants `DROP` instead, `DROP COLUMN`, `DROP CHECK`, `DROP PRIMARY KEY`, and `DROP FOREIGN KEY`. Keep this in mind, the concept from lecture is exactly right, only the keyword MySQL happens to use is different.

In [ ]:
cursor.execute("ALTER TABLE item_scratch DROP COLUMN Item_Notes;")
cursor.execute("ALTER TABLE item_scratch DROP CHECK item_scratch_price_chk;")
cursor.execute("ALTER TABLE item_scratch DROP PRIMARY KEY;")
conn.commit()

cursor.execute("DESCRIBE item_scratch;")
for row in cursor.fetchall():
    print(row)

### Renaming and Dropping

Same story as `DELETE` versus `DROP`. The slides show `ALTER TABLE item RENAME title TO item_title;` for a column and `ALTER TABLE order_line RENAME line_item;` for a table. MySQL wants `RENAME COLUMN old TO new` for a column, and `RENAME TO new_name` for the table itself.

In [ ]:
cursor.execute("ALTER TABLE item_scratch RENAME COLUMN Item_Description TO Item_Desc;")
cursor.execute("ALTER TABLE item_scratch RENAME TO item_scratch_renamed;")
conn.commit()

cursor.execute("DESCRIBE item_scratch_renamed;")
for row in cursor.fetchall():
    print(row)

Finally, `DROP TABLE` removes a table entirely, and MySQL does not ask for confirmation, so we clean up our scratch table now that we are done with it. This leaves the real schema, *customer* through *shipment_info*, untouched and ready for the next tutorial.

In [ ]:
cursor.execute("DROP TABLE item_scratch_renamed;")
conn.commit()
print("item_scratch_renamed dropped.")

### Closing Up

The full schema, nine tables and every foreign key between them, is now sitting in `iem4723_mysql_tutorial`, ready for Tutorial 2. We close *cursor* and *conn* the same way we did at the end of the introduction notebook.

In [ ]:
cursor.close()
conn.close()